# 03 — Collections & Generics

**What you'll learn:**

- The `Collection` hierarchy: `List`, `Set`, `Queue`, `Map`
- `ArrayList` vs `LinkedList` — when each is the right choice
- `HashSet` vs `TreeSet` vs `LinkedHashSet`
- `HashMap` vs `TreeMap` vs `LinkedHashMap`
- `ArrayDeque` as the go-to queue/stack
- Iterating with enhanced `for` and `Iterator`
- Immutable collection factories: `List.of`, `Set.of`, `Map.of`
- Generics — why type parameters exist
- Generic classes and generic methods
- Bounded type parameters with `extends`
- Wildcards: `? extends T` (covariance) and `? super T` (contravariance)
- The PECS mnemonic — Producer Extends, Consumer Super
- Type erasure — what the JVM actually sees at runtime

Collections are how Java represents groups of values. Generics are how Java describes the *type* inside those groups without copy-pasting a class per element type. The two go hand in hand — you almost never see one without the other in real code.

## The Collection hierarchy

Java's collection framework lives in `java.util`. At the top is the `Collection<E>` interface — anything that holds elements. It branches into three big families:

```text
                 Iterable<E>
                     |
                 Collection<E>
                /    |     \
           List<E>  Set<E>  Queue<E>
              |       |       |
         +----+---+   |    Deque<E>
         |        |   |       |
     ArrayList LinkedList    ArrayDeque
                              |
                          (HashSet, TreeSet, ...)
```

Separate from `Collection` sits `Map<K,V>` — it associates keys with values rather than just holding elements.

- **`List`** — ordered, indexable, allows duplicates.
- **`Set`** — no duplicates, may or may not be ordered.
- **`Queue` / `Deque`** — FIFO or double-ended access.
- **`Map`** — key-to-value lookup.

Each interface has multiple implementations, each tuned for different performance trade-offs.

## `List` — ordered, indexed, allows duplicates

The two implementations you'll meet 99% of the time are `ArrayList` and `LinkedList`.

- **`ArrayList`** — backed by a resizing array. O(1) random access by index, O(1) append amortised, O(n) insert/remove in the middle. **This is your default.**
- **`LinkedList`** — doubly-linked list. O(1) insert/remove at either end, O(n) random access. Rarely the right choice — use `ArrayDeque` for queue/stack needs instead.

In [ ]:
import java.util.*;

List<String> names = new ArrayList<>();
names.add("alice");
names.add("bob");
names.add("carol");

names.size();          // 3
names.get(1);          // "bob"  — O(1)
names.indexOf("carol");// 2
names.contains("bob"); // true
names.set(0, "alex");  // replace at index
names.remove(2);       // remove by index
names.remove("alex");  // remove first occurrence

// iterate
for (var name : names) {
    System.out.println(name);
}

## `Set` — no duplicates

Adding a value that's already present is a no-op; the set stays the same size. The three implementations differ in *iteration order*.

- **`HashSet`** — O(1) add/remove/contains on average; iteration order is unspecified. Default choice.
- **`LinkedHashSet`** — same performance as `HashSet`, but iterates in **insertion order**. Use when you need duplicate-removal that preserves the order elements first appeared.
- **`TreeSet`** — O(log n) operations; iterates in **sorted order**. Use when you need the set to be sorted at all times.

In [ ]:
Set<Integer> seen = new HashSet<>();
seen.add(1);
seen.add(2);
seen.add(1);        // duplicate — ignored
seen.size();        // 2
seen.contains(2);   // true

// sorted iteration
Set<Integer> sorted = new TreeSet<>(List.of(3, 1, 4, 1, 5, 9, 2, 6));
// iterates 1, 2, 3, 4, 5, 6, 9

// insertion-order iteration
Set<String> ordered = new LinkedHashSet<>();
ordered.add("c"); ordered.add("a"); ordered.add("b");
// iterates c, a, b

## `Map` — key-to-value lookup

A `Map<K,V>` stores key-value pairs. The same three flavours apply as with `Set`:

- **`HashMap`** — O(1) average lookup; iteration order unspecified. Default.
- **`LinkedHashMap`** — preserves insertion order. Useful when you want predictable iteration.
- **`TreeMap`** — keys kept sorted; O(log n) operations.

In [ ]:
Map<String, Integer> ages = new HashMap<>();
ages.put("alice", 30);
ages.put("bob", 25);
ages.put("carol", 28);

ages.get("alice");           // 30
ages.getOrDefault("dan", 0); // 0  — fallback if absent
ages.containsKey("bob");     // true
ages.size();                 // 3

// iterate keys, values, or entries
for (var name : ages.keySet())     { System.out.println(name); }
for (var age  : ages.values())     { System.out.println(age); }
for (var entry : ages.entrySet()) {
    System.out.println(entry.getKey() + " -> " + entry.getValue());
}

// modern helpers
ages.putIfAbsent("dan", 22);                // only puts if not already present
ages.merge("alice", 1, Integer::sum);       // ages["alice"] += 1
ages.computeIfAbsent("eve", k -> 0);        // lazily insert a default

## `Queue` and `Deque`

A `Queue<E>` is FIFO — first in, first out. A `Deque<E>` is a double-ended queue — you can add or remove from either end, so it also serves as a stack.

The right implementation in almost every case is **`ArrayDeque`**. It's faster than `LinkedList` for both queue and stack workloads and avoids `Stack`, which is a legacy synchronised class you should not use.

In [ ]:
Deque<Integer> stack = new ArrayDeque<>();
stack.push(1);
stack.push(2);
stack.push(3);
stack.pop();      // 3
stack.peek();     // 2

Queue<String> queue = new ArrayDeque<>();
queue.offer("first");
queue.offer("second");
queue.poll();     // "first"
queue.peek();     // "second"

## Immutable collection factories

Since Java 9, every collection interface has static `of` methods that return **immutable** instances. Use them for constants, configuration, and any collection you don't intend to modify.

In [ ]:
List<String>        roles  = List.of("admin", "editor", "viewer");
Set<Integer>        prime  = Set.of(2, 3, 5, 7, 11);
Map<String, Integer> codes = Map.of(
    "OK", 200,
    "NOT_FOUND", 404,
    "SERVER_ERROR", 500
);

// roles.add("guest");        // UnsupportedOperationException
// codes.put("FORBIDDEN", 403); // UnsupportedOperationException

// Map.ofEntries lets you go past the 10-entry limit of Map.of
Map<String, Integer> longer = Map.ofEntries(
    Map.entry("a", 1),
    Map.entry("b", 2),
    Map.entry("c", 3)
);

These collections are **structurally immutable**: you can't add, remove, or replace elements. They are also unmodifiable views of the data you passed in — `List.of` with no arguments returns the same shared empty list every time. Two big benefits: they are safe to share across threads, and they make defensive copies unnecessary.

## Iterating

The **enhanced `for`** (often called *for-each*) is the idiomatic way to iterate any `Iterable`, including every collection above and plain arrays.

In [ ]:
var names = List.of("alice", "bob", "carol");

// enhanced for — preferred
for (var name : names) {
    System.out.println(name);
}

// classic Iterator — needed when you want to remove during iteration
var it = new ArrayList<>(names).iterator();
while (it.hasNext()) {
    var name = it.next();
    if (name.startsWith("a")) {
        it.remove();              // safe — can't do this with enhanced for
    }
}

// indexed loop — when you really need the index
for (int i = 0; i < names.size(); i++) {
    System.out.println(i + ": " + names.get(i));
}

A subtle rule: you cannot modify a collection during an enhanced-`for` over it — doing so throws `ConcurrentModificationException`. Use the explicit `Iterator` and its `remove()` method, or iterate over a copy.

## Generics — what and why

Notice that every collection above was declared with an angle-bracket type, like `List<String>` or `Map<String, Integer>`. That `<E>` is a **type parameter** — a placeholder filled in at each use site.

Generics give you two things at once:

1. **Compile-time type safety** — the compiler stops you from adding an `Integer` to a `List<String>`.
2. **No casting at retrieval** — `names.get(0)` already returns a `String`, not an `Object` you have to cast.

Before generics (Java 1.4 and earlier), collections held raw `Object` references. Every retrieval needed an explicit cast, and the cast could fail at runtime. Generics turn that runtime error into a compile-time error.

In [ ]:
// raw type — legacy, avoid
List rawList = new ArrayList();
rawList.add("hello");
rawList.add(42);                    // compiles, but is wrong
String s = (String) rawList.get(1); // ClassCastException at runtime

// generic — the compiler enforces the type
List<String> safe = new ArrayList<>();
safe.add("hello");
// safe.add(42);    // COMPILE ERROR
String t = safe.get(0);    // no cast needed

## Writing a generic class

You can declare your own generic types by listing type parameters in angle brackets after the class name.

In [ ]:
public class Box<T> {
    private T value;

    public Box(T value) { this.value = value; }

    public T get()                 { return value; }
    public void set(T newValue)    { this.value = newValue; }
}

var intBox = new Box<Integer>(42);
intBox.get();                   // 42 — return type is Integer

var strBox = new Box<String>("hello");
strBox.get();                   // "hello" — return type is String

// the diamond operator <> lets the compiler infer the type from context
Box<Double> piBox = new Box<>(3.14);

A single class definition serves any element type. The compiler enforces the right type per instantiation.

## Generic methods

A method can declare its own type parameter, independent of its enclosing class. The type parameter goes before the return type.

In [ ]:
public class Util {
    // <T> declares a method-level type parameter
    public static <T> T firstOrNull(List<T> list) {
        return list.isEmpty() ? null : list.get(0);
    }

    public static <K, V> Map<V, K> invert(Map<K, V> source) {
        var result = new HashMap<V, K>();
        for (var entry : source.entrySet()) {
            result.put(entry.getValue(), entry.getKey());
        }
        return result;
    }
}

Util.firstOrNull(List.of("alice", "bob"));   // "alice"  (T = String)
Util.firstOrNull(List.of(1, 2, 3));          // 1        (T = Integer)
Util.invert(Map.of("a", 1, "b", 2));         // {1=a, 2=b}

## Bounded type parameters

Sometimes you need a guarantee that the type parameter is a subtype of something specific — for example, "any `Number`". Use `extends` to constrain it.

In [ ]:
public class Stats {
    // T must be Number or a subtype — gives us doubleValue()
    public static <T extends Number> double sum(List<T> list) {
        double total = 0;
        for (var n : list) {
            total += n.doubleValue();
        }
        return total;
    }
}

Stats.sum(List.of(1, 2, 3));            // 6.0   — T = Integer
Stats.sum(List.of(1.5, 2.5, 3.0));      // 7.0   — T = Double
// Stats.sum(List.of("a", "b"));        // COMPILE ERROR — String isn't a Number

You can also chain bounds with `&`. `<T extends Number & Comparable<T>>` means T must be both a `Number` and `Comparable` to itself.

## Wildcards: `? extends T` and `? super T`

Here is where generics get subtle. Even though `Integer extends Number`, **`List<Integer>` is *not* a subtype of `List<Number>`**. Java generics are *invariant* by default. A method declared `void process(List<Number> list)` will not accept a `List<Integer>`.

This is correct! If `List<Integer>` *were* assignable to `List<Number>`, then `process` could `add(3.14)` to it and your `List<Integer>` would suddenly contain a `Double`.

**Wildcards** let you opt into covariance or contravariance at the use site:

- **`List<? extends T>`** — *covariance*. "A list of T or any subtype." You can read T values out, but you can't add anything (except `null`) because the compiler doesn't know the exact element type.
- **`List<? super T>`** — *contravariance*. "A list of T or any supertype." You can add T values, but reading gives you `Object` because the compiler can't guarantee a more specific type.

In [ ]:
// ? extends Number — covariant, read-only
public static double sumAll(List<? extends Number> list) {
    double total = 0;
    for (var n : list) total += n.doubleValue();   // reading T works
    // list.add(1);   // COMPILE ERROR — can't write to ? extends
    return total;
}

sumAll(List.of(1, 2, 3));               // works — List<Integer>
sumAll(List.of(1.5, 2.5));              // works — List<Double>


// ? super Integer — contravariant, write-friendly
public static void fillWithIntegers(List<? super Integer> dest) {
    dest.add(1);     // adding Integer works
    dest.add(2);
    // Integer x = dest.get(0);   // COMPILE ERROR — read returns Object
}

List<Number> nums    = new ArrayList<>();
List<Object> objects = new ArrayList<>();
fillWithIntegers(nums);     // works
fillWithIntegers(objects);  // works — Object is a supertype of Integer

## PECS — Producer Extends, Consumer Super

The mnemonic Joshua Bloch coined in *Effective Java*:

- If a parameter **produces** values for you to consume, declare it `? extends T`.
- If a parameter **consumes** values you provide, declare it `? super T`.
- If it does both, leave it as plain `T` (invariant).

This is exactly the rule the JDK uses. Look at `Collections.copy`:

```java
public static <T> void copy(List<? super T> dest, List<? extends T> src) { ... }
```

The destination *consumes* T values → `? super T`. The source *produces* T values → `? extends T`.

## Type erasure — what the JVM actually sees

Java generics are a **compile-time** feature. At runtime, `List<String>` and `List<Integer>` are both just `List`. The type parameter is **erased** during compilation, replaced with the bound (or `Object` if unbounded).

Three practical consequences:

- **You can't write `new T[10]`** — the JVM doesn't know what `T` is at runtime.
- **You can't write `if (x instanceof List<String>)`** — the type parameter isn't available. Use `instanceof List<?>` instead.
- **Two methods that differ only in their type parameters compile to the same signature** — `void f(List<String>)` and `void f(List<Integer>)` would clash.

Erasure exists for backwards compatibility — old code that uses raw `List` and new code that uses `List<String>` are the same `List` at the JVM level, so they interoperate. The trade is that generics carry no runtime type information.

In [ ]:
List<String> ss = new ArrayList<>();
List<Integer> ii = new ArrayList<>();

ss.getClass() == ii.getClass();    // true — both are ArrayList at runtime

// instanceof with parameterized type uses a wildcard
Object obj = ss;
if (obj instanceof List<?> list) {     // OK
    System.out.println("got a list of size " + list.size());
}
// if (obj instanceof List<String>)    // COMPILE ERROR

## What's next

You can now reach for the right collection for the job, define your own generic types, and read the wildcard signatures that pepper the JDK. Notebook 04 turns these collections into the engine of modern Java: **lambdas**, **functional interfaces**, and the **Stream API** that lets you `filter`, `map`, and `reduce` your way through data the way Python comprehensions or Spark's RDD operators do. We will also meet `Optional`, the type-safe alternative to `null`.